# Fairness Over Time — Phase 3: Diagnose (Texas, 2014–2017)

**RQ2: Why does fairness change?**

Phase 2 found that the frozen 2014 models lose about 1.2 points of accuracy by 2017 while AUC stays flat, and that their fairness gaps (EOD) rise slowly. This notebook looks for the cause with four analyses, each answering one question:

| Section | Question | Method |
|---|---|---|
| 1. Label drift | Did the *outcome* change? How much is inflation? | Fixed vs. inflation-indexed $50K threshold |
| 2. Feature drift | Did the *inputs* change? | PSI and KS against 2014 training data |
| 3. SHAP over time | Do the models *use* the inputs differently? | Mean \|SHAP\| per feature, per year |
| 4. Group TPR | *Which groups* drive the EOD change? | True positive rate by race/ethnicity × sex |

The models are never retrained here; every analysis reads the frozen models or the data.

In [1]:
%load_ext autoreload
%autoreload 2

import time
import warnings
import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import accuracy_score, roc_auc_score

from src.data import DATA_DIR, ARTIFACTS_DIR, RESULTS_DIR, REPORTS_DIR, METRICS_PATH, CI_PATH, LABEL_DRIFT_PATH, DRIFT_PATH, SHAP_PATH, BACKGROUND_PATH, load_acs, make_race_eth, split_data, save_split, load_split
from src.preprocess import NUMERIC_COLS, ORDINAL_COLS, NOMINAL_COLS, build_preprocessor
from src.models import make_lr, make_xgb, make_pipeline_for, fit_mitigated, load_models, MODEL_INFO
from src.fairness import (make_sensitive, group_metrics, fairness_summary, fairness_long,
                          bootstrap_gaps, evaluate_all, bootstrap_all, save_results)
from src.drift import label_rates, label_rates_by_group, feature_drift

warnings.filterwarnings("ignore", module="fairlearn")

import shap

## 1. Label drift: inflation or real income growth?

From Phase 2, the share of Texans earning over $50K rose from 31.6% to 35.4% (+3.8 points). The threshold is a fixed $50,000, so two things can push people over it:
- **Inflation:** prices rise, so the same real pay is a bigger dollar number.
- **Real growth:** people actually earn more, after inflation.

First, two checks on how income is recorded.

In [2]:
#load one year
X, y, meta =load_acs(2017)
models = load_models()

print(((meta["income"] > 50_000).astype(int) == y.squeeze("columns")).all())
print(meta["ADJINC"].unique()) 

True
[1011189]


**Both checks pass:**
- The label is exactly `income > 50,000` for every row, using raw, unadjusted income (`PINCP`).
- `ADJINC` has a single value per year (2017: 1.011189, stored with 6 implied decimals). It puts all incomes *within* one survey year into that year's dollars, because people report income for different 12-month windows. It does not make years comparable, and every year's factor is about 1.01, so it is not applied.

**CPI-U is used to compare dollars across years** (BLS annual averages, U.S. city average).

### Method: fixed vs. indexed threshold

For each year, compare two positive rates:
- **Fixed rate:** share earning over $50,000 (the actual label).
- **Indexed rate:** share earning over $50,000 × CPI(year) / CPI(2014), i.e. $50K in 2014 dollars: about $50,059 in 2015, $50,691 in 2016 and $51,771 in 2017.

Then:
- **Inflation part** = fixed rate − indexed rate (people who cross $50K only because of price changes)
- **Real-growth part** = indexed rate − 2014 rate

The two parts add up to the total change. Check: in 2014 both rates are equal (31.6%), as they should be.

In [3]:
YEARS = [2014, 2015, 2016, 2017]
metas = {year: load_acs(year)[2] for year in YEARS}

rates = pd.DataFrame([label_rates(metas[y]["income"], y) for y in YEARS])

base_rate = rates.loc[rates["year"] == 2014, "fixed_rate"].item()
rates["inflation_part"] = rates["fixed_rate"] - rates["indexed_rate"]
rates["real_growth"] = rates['indexed_rate'] - base_rate

rates

,year,fixed_rate,indexed_rate,inflation_part,real_growth
0,2014,0.316352,0.316352,0.000000,0.000000
1,2015,0.331646,0.331328,0.000318,0.014976
2,2016,0.341508,0.339777,0.001732,0.023425
3,2017,0.353854,0.347729,0.006125,0.031378


### Finding: the label drift is mostly real income growth

| Year | Fixed rate | Indexed rate | Inflation part | Real-growth part |
|---|---|---|---|---|
| 2014 | 31.64% | 31.64% | 0 | 0 |
| 2015 | 33.16% | 33.13% | 0.03 pts | 1.50 pts |
| 2016 | 34.15% | 33.98% | 0.17 pts | 2.34 pts |
| 2017 | 35.39% | 34.77% | 0.61 pts | 3.14 pts |

**About 84% of the 2014 → 2017 rise is real income growth; about 16% is inflation.** Inflation explains almost nothing in 2015, when prices barely rose (+0.1%, held down by the oil-price collapse).

**What this means:** the 2014 cutoff goes out of date mainly because the population really changed, not just because $50K is worth less.

**Caveat:** reported incomes cluster on round numbers (many people report exactly $50,000, which counts as negative under both thresholds), so the inflation share is likely a lower bound. The conclusion does not change.

In [4]:
label_drift = pd.concat(
    [label_rates_by_group(metas[y], y) for y in YEARS], ignore_index=True
)
save_results(label_drift, LABEL_DRIFT_PATH)

# Readable check table (not saved): one row per attribute/group/year
check = label_drift.pivot(index=["attribute", "group", "year"],
                          columns="metric", values="value")
check["inflation_part"] = check["fixed_rate"] - check["indexed_rate"]
check["real_part"] = check["indexed_rate"] - check.groupby(level=["attribute", "group"])["fixed_rate"].transform("first")
check.round(4)

metric                   fixed_rate  indexed_rate  inflation_part  real_part
attribute group    year                                                     
overall   all      2014      0.3164        0.3164          0.0000     0.0000
                   2015      0.3316        0.3313          0.0003     0.0150
                   2016      0.3415        0.3398          0.0017     0.0234
                   2017      0.3539        0.3477          0.0061     0.0314
race_eth  Hispanic 2014      0.1692        0.1692          0.0000     0.0000
                   2015      0.1829        0.1828          0.0001     0.0136
                   2016      0.1887        0.1875          0.0012     0.0184
                   2017      0.2002        0.1957          0.0044     0.0266
          NH Asian 2014      0.4287        0.4287          0.0000     0.0000
                   2015      0.4326        0.4326          0.0000     0.0039
                   2016      0.4470        0.4462          0.0008     0.0175
                   2017      0.4732        0.4706          0.0027     0.0419
          NH Black 2014      0.2190        0.2190          0.0000     0.0000
                   2015      0.2333        0.2332          0.0001     0.0143
                   2016      0.2483        0.2466          0.0017     0.0276
                   2017      0.2671        0.2615          0.0056     0.0425
          NH Other 2014      0.2793        0.2793          0.0000     0.0000
                   2015      0.3100        0.3087          0.0013     0.0294
                   2016      0.3256        0.3247          0.0008     0.0454
                   2017      0.3173        0.3132          0.0041     0.0339
          NH White 2014      0.4109        0.4109          0.0000     0.0000
                   2015      0.4295        0.4290          0.0005     0.0182
                   2016      0.4416        0.4394          0.0022     0.0285
                   2017      0.4525        0.4449          0.0076     0.0340
sex       Female   2014      0.2279        0.2279          0.0000     0.0000
                   2015      0.2427        0.2423          0.0004     0.0145
                   2016      0.2591        0.2573          0.0018     0.0294
                   2017      0.2720        0.2658          0.0062     0.0380
          Male     2014      0.3935        0.3935          0.0000     0.0000
                   2015      0.4090        0.4087          0.0003     0.0153
                   2016      0.4135        0.4118          0.0016     0.0184
                   2017      0.4253        0.4192          0.0061     0.0257

### Finding: groups grow at different speeds

Change 2014 → 2017 in the share earning over $50K (percentage points):

| Group | Total | Inflation | Real growth |
|---|---|---|---|
| Female | +4.42 | 0.62 | 3.80 |
| Male | +3.18 | 0.61 | 2.57 |
| NH Black | +4.81 | 0.56 | 4.25 |
| NH Asian | +4.46 | 0.27 | 4.19 |
| NH White | +4.16 | 0.76 | 3.40 |
| Hispanic | +3.10 | 0.44 | 2.66 |

- **The male − female gap narrows (16.6 → 15.3 points) through real growth:** women gained more than men.
- **The NH White − Hispanic gap widens (24.2 → 25.2):** Hispanic workers had the slowest real growth of any group, and about 0.3 points of the widening comes from the fixed threshold itself.
- The inflation part is largest where many people sit just below $50K (NH White) and smallest where most are far from it (NH Asian above, Hispanic below).
- NH Other (about 2,200 people) is left out of the interpretation because it is too noisy.

These are changes in the **true** group rates, before any model is involved. That is why DPD can move even if a model works as before, and why EOD is the main fairness measure.

Saved to `artifacts/results/label_drift.csv` (64 rows).

## 2. Feature drift: did the inputs change?

If the people in later years look different (older, more educated, different jobs), the models would be applied to inputs unlike the data they learned from.

- **Baseline = the 2014 training rows** (76,223), which is what the models learned from.
- Compared with **all rows** of 2015, 2016 and 2017.
- **Noise floor = the 2014 test rows.** Same year, so no real change: any drift measured there is pure sampling noise. Later years are judged against it.

In [5]:
train_idx, val_idx, test_idx = load_split()

Xs = {y: load_acs(y)[0] for y in YEARS}

X_base = Xs[2014].loc[train_idx]

comparisons = {
    2014: Xs[2014].loc[test_idx],
    2015: Xs[2015],
    2016: Xs[2016],
    2017: Xs[2017],
}

### Two drift measures

**PSI (Population Stability Index)** compares how a feature's values are spread across bins in two datasets:

PSI = Σ over bins of (current share − base share) × ln(current share / base share)

- It is 0 when the shares are identical and grows when shares move, especially when a rare bin becomes common or the reverse.
- Bins are fitted on 2014 training data only, then applied to every year (the same rule as the preprocessing). Age and hours worked use deciles; categorical features use one bin per category with at least 30 training rows, and the rest go to "other".
- Rule of thumb from credit scoring: < 0.1 small, 0.1–0.25 moderate, > 0.25 large. This is a guide, not a test.

**KS statistic** (for ordered features only: age, hours, education): the largest gap between the two cumulative distributions. It is used as a size measure; its p-value is not reported, because with about 130K rows per year almost any difference would be "significant".

In [6]:
from src.drift import make_bins, assign_bins, psi

b = make_bins(X_base["AGEP"], numeric=True)
base_b = assign_bins(X_base["AGEP"], b, numeric=True)

print("self    :", psi(base_b, base_b))                                                   # must be 0.0
print("noise   :", psi(base_b, assign_bins(comparisons[2014]["AGEP"], b, numeric=True)))  # 2014 test
print("2017    :", psi(base_b, assign_bins(comparisons[2017]["AGEP"], b, numeric=True)))
print(base_b.value_counts(normalize=True).sort_index())                                   # ~10% per bin?

self    : 0.0
noise   : 0.00024580676119014244
2017    : 0.0015750969071915218
AGEP
(-inf, 23.0]    0.113430
(23.0, 28.0]    0.099353
(28.0, 33.0]    0.103879
(33.0, 38.0]    0.101374
(38.0, 43.0]    0.100980
(43.0, 47.0]    0.082810
(47.0, 52.0]    0.108038
(52.0, 57.0]    0.108511
(57.0, 62.0]    0.087651
(62.0, inf]     0.093974
Name: proportion, dtype: float64


**The PSI function passes its sanity checks:** a dataset compared with itself gives exactly 0; the 2014 test set (noise floor) gives 0.0002; 2017 gives 0.0016. The age deciles each hold about 8–11% of people instead of exactly 10%, because ages are whole numbers, so many people share the same edge value.

In [7]:
drift = pd.concat(
    [feature_drift(X_base, X_cur, year) for year, X_cur in comparisons.items()],
    ignore_index=True,
)
print(len(drift))
save_results(drift, DRIFT_PATH)

# Readable view: one row per feature/metric, one column per year
view = drift.pivot(index=["metric", "feature"], columns="year", values="value")
view["x_noise_2017"] = view[2017] / view[2014]   # 2017 drift as a multiple of the noise floor
view.sort_values(["metric", 2017], ascending=[True, False]).round(4)

52


year               2014    2015    2016    2017  x_noise_2017
metric feature                                               
KS     SCHL      0.0028  0.0055  0.0170  0.0279        9.9943
       WKHP      0.0027  0.0071  0.0097  0.0097        3.5314
       AGEP      0.0041  0.0042  0.0068  0.0091        2.2236
PSI    OCCP      0.0161  0.0101  0.0122  0.0167        1.0373
       SCHL      0.0009  0.0011  0.0021  0.0052        6.0046
       POBP      0.0046  0.0027  0.0034  0.0048        1.0446
       race_eth  0.0000  0.0005  0.0010  0.0023     4069.4706
       AGEP      0.0002  0.0003  0.0007  0.0016        6.4079
       WKHP      0.0002  0.0005  0.0009  0.0014        9.0871
       RELP      0.0010  0.0008  0.0011  0.0011        1.0593
       COW       0.0008  0.0005  0.0006  0.0007        0.8426
       MAR       0.0001  0.0002  0.0007  0.0006        7.3367
       SEX       0.0000  0.0000  0.0000  0.0000       28.9787

### Finding: the inputs barely changed

**Every PSI is below 0.02**, far under the 0.1 "small" line. Compared with the noise floor:

| Group | Features | 2017 vs. noise |
|---|---|---|
| Real and rising every year | SCHL (education) | KS 10× noise: more Texans with higher degrees |
| Real but small | WKHP, AGEP, MAR | 2–9× noise, but tiny in absolute terms |
| Indistinguishable from noise | OCCP, POBP, RELP, COW | about 1× |
| Small, rising | race_eth | 0.0023 in 2017 (matches Phase 2's population shares) |
| No change | SEX | 0 |

**Note on the ratio column:** for SEX and race_eth the noise floor is zero by construction, because the 2014 split was stratified on them. Their ratios (29, 4069) are meaningless; judge them by their raw values.

- No detectable change in the occupation mix, even through the 2014–2016 oil crash: the shock changed pay more than the mix of jobs, at least at this level of detail.

**Conclusion with Section 1: this is mainly concept drift.** The inputs are almost stable, but the label rose 3.8 points, mostly through real income growth. The same features now go with a higher chance of earning over $50K. This explains Phase 2: AUC stays flat (the inputs still rank people correctly), while accuracy falls (the same scores now correspond to more true positives).

Saved to `artifacts/results/drift_over_time.csv` (52 rows).

## 3. SHAP over time: do the models use the features differently?

If the models started relying on different features in later years, that could explain the fairness change. SHAP measures how much each feature pushes one person's prediction up or down, in log-odds.

**Method:**
- **Unmitigated LR and XGBoost only.** The mitigated models are weighted votes of several thresholded models, so TreeSHAP does not apply cleanly.
- **Explainers:** XGBoost uses `TreeExplainer` (exact and fast). LR uses `LinearExplainer` with a background of 500 rows from 2014 training data, saved to `artifacts/background_sample.csv` so the app explains predictions the same way.
- **Encoded features summed back:** SHAP values were computed on the 456 encoded features and summed back to the original 10 features (exact for LR, a standard approximation for XGBoost).
- **1,000 rows per year** (seed 42): 2014 = test set as in Phase 2, 2015–2017 = full years.
- **Importance = mean |SHAP|** per feature per year. Years are compared within a model; the two models are compared only by ranking, because their reference points differ.

In [8]:
N_SHAP = 1_000
N_BACKGROUND = 500
SEED = 42

# 1,000 rows per year to explain. comparisons[2014] is the 2014 TEST set, as in Phase 2.
shap_X = {year: X_cur.sample(N_SHAP, random_state=SEED) for year, X_cur in comparisons.items()}

# Background: 500 raw rows from 2014 TRAIN. Used by LR's explainer now and by the app later.
background = X_base.sample(N_BACKGROUND, random_state=SEED)
background.to_csv(BACKGROUND_PATH, index=False)

print({y: len(x) for y, x in shap_X.items()})

{2014: 1000, 2015: 1000, 2016: 1000, 2017: 1000}


In [9]:
#run shap on xgboost in 2014
from src.explain import explain
shap_14 = explain(models["xgb"], shap_X[2014], background)

print(shap_14.shape)                                         # expect (1000, 10)
shap_14.abs().mean().sort_values(ascending=False)

(1000, 10)


AGEP        0.636628
SCHL        0.618378
WKHP        0.588411
OCCP        0.547921
SEX         0.299521
RELP        0.224680
race_eth    0.210652
MAR         0.136367
COW         0.123498
POBP        0.048417
dtype: float32

In [10]:
#run shap on linear regression in 2014
shap_lr14 = explain(models["lr"], shap_X[2014], background)

# Additivity: per person, sum of SHAP = log-odds − average background log-odds
pre, lr = models["lr"].named_steps["pre"], models["lr"].named_steps["model"]
expected = lr.decision_function(pre.transform(shap_X[2014])) \
           - lr.decision_function(pre.transform(background)).mean()
print(np.allclose(shap_lr14.sum(axis=1), expected, atol=1e-6))   # expect True

shap_lr14.abs().mean().sort_values(ascending=False)

True


OCCP        0.876476
SCHL        0.452687
AGEP        0.443841
WKHP        0.424264
RELP        0.385836
SEX         0.336718
MAR         0.198544
race_eth    0.152090
COW         0.141146
POBP        0.086724
dtype: float64

**Checks pass:** each result has 1,000 rows × 10 features, and for LR the SHAP values add up exactly to each person's log-odds minus the average background log-odds (`True`).

In [11]:
from src.explain import shap_importance

SHAP_MODELS = {"lr": "LR", "xgb": "XGBoost"}
shap_values = {}                                   # keep per-person SHAP for the optional group view
rows = []
for year, X_s in shap_X.items():
    for name, label in SHAP_MODELS.items():
        sv = explain(models[name], X_s, background)
        shap_values[(year, label)] = sv
        rows.append(shap_importance(sv, year, label))

shap_imp = pd.concat(rows, ignore_index=True)
print(len(shap_imp))                               # expect 4 × 2 × 10 × 2 = 160
save_results(shap_imp, SHAP_PATH)

view = shap_imp[shap_imp["metric"] == "mean_abs_shap"].pivot(
    index=["model", "feature"], columns="year", values="value")
view["change_14_17"] = view[2017] - view[2014]
view.sort_values(["model", 2014], ascending=[True, False]).round(3)

160


year               2014   2015   2016   2017  change_14_17
model   feature                                           
LR      OCCP      0.876  0.871  0.877  0.852        -0.025
        SCHL      0.453  0.473  0.481  0.482         0.030
        AGEP      0.444  0.452  0.451  0.436        -0.008
        WKHP      0.424  0.403  0.411  0.437         0.012
        RELP      0.386  0.385  0.387  0.376        -0.010
        SEX       0.337  0.336  0.337  0.337        -0.000
        MAR       0.199  0.203  0.205  0.200         0.001
        race_eth  0.152  0.151  0.150  0.150        -0.002
        COW       0.141  0.131  0.135  0.137        -0.004
        POBP      0.087  0.087  0.089  0.088         0.001
XGBoost AGEP      0.637  0.620  0.616  0.588        -0.049
        SCHL      0.618  0.639  0.634  0.640         0.022
        WKHP      0.588  0.539  0.555  0.589         0.000
        OCCP      0.548  0.542  0.545  0.534        -0.014
        SEX       0.300  0.300  0.299  0.302         0.002
        RELP      0.225  0.218  0.216  0.223        -0.002
        race_eth  0.211  0.212  0.211  0.207        -0.003
        MAR       0.136  0.141  0.139  0.139         0.003
        COW       0.123  0.116  0.120  0.121        -0.002
        POBP      0.048  0.050  0.050  0.050         0.002

In [12]:
shap_values[(2017, "XGBoost")].abs().std() / np.sqrt(1000)

AGEP        0.019482
WKHP        0.018556
SCHL        0.010603
COW         0.003880
MAR         0.003192
OCCP        0.014130
POBP        0.002332
RELP        0.006856
SEX         0.002800
race_eth    0.002719
dtype: float32

**How much noise is in 1,000 rows?** The standard error of mean |SHAP| for XGBoost in 2017 is up to about 0.02 (AGEP, WKHP). A difference between two years therefore has an SE of about 0.02 × √2 ≈ 0.03, so changes up to about 0.05 are within sampling noise (for example, XGBoost's WKHP goes 0.588 → 0.539 → 0.589).

### Finding: SHAP importance is flat over time

- **All changes from 2014 to 2017 are ≤ 0.05, and the ranking of features does not change in either model.** The frozen models use the features the same way every year, so the fairness change is not caused by the models reacting to different inputs.
- **SCHL (education) rises slightly in both models** (in LR every year), in line with the education drift from Section 2.
- **The two models rely on different things:** LR is dominated by occupation (OCCP 0.88, about twice the next feature), while XGBoost spreads its weight evenly over age, education, hours and occupation (0.53–0.64).
- **Both use sex directly** (about 0.3) and race/ethnicity (0.15–0.21). Household role (RELP, MAR) may also act as a proxy for sex.

Saved to `artifacts/results/shap_over_time.csv` (160 rows).

## 4. Which groups drive the change in EOD?

EOD is the largest gap in true positive rate (TPR) or false positive rate between any two race/ethnicity × sex groups. Looking at each group's TPR in 2014 and 2017 shows which groups move and whether the gap opens at the top, at the bottom, or both. Taken from `metrics_over_time.csv` for unmitigated LR, unmitigated XGBoost and mitigated LR (whose EOD rise in Phase 2 was hard to judge).

In [13]:
m = pd.read_csv(METRICS_PATH)
tpr = m[(m["attribute"] == "race_eth_x_sex") & (m["metric"] == "TPR") & (m["year"].isin([2014, 2017]))].copy()
tpr["name"] = tpr["model"] + np.where(tpr["mitigated"], " mitigated", "")
tpr = tpr[tpr["name"].isin(["LR", "XGBoost", "LR mitigated"])]

t = tpr.pivot(index="group", columns=["name", "year"], values="value")
tpr_2014 = t.xs(2014, axis=1, level="year")
change   = t.xs(2017, axis=1, level="year") - tpr_2014

print("TPR 2014");          display(tpr_2014.round(3))
print("Change 2014 → 2017"); display(change.round(3).sort_values("LR"))

TPR 2014


name,LR,LR mitigated,XGBoost
group,,,
Hispanic × Female,0.323,0.622,0.368
Hispanic × Male,0.531,0.570,0.469
NH Asian × Female,0.648,0.654,0.726
NH Asian × Male,0.847,0.705,0.830
NH Black × Female,0.393,0.674,0.492
NH Black × Male,0.547,0.595,0.519
NH Other × Female,0.442,0.721,0.465
NH Other × Male,0.691,0.642,0.617
NH White × Female,0.572,0.632,0.580


Change 2014 → 2017


name,LR,LR mitigated,XGBoost
group,,,
NH Other × Female,-0.066,-0.175,0.009
Hispanic × Male,-0.061,-0.051,-0.045
NH Black × Female,-0.033,-0.035,-0.020
Hispanic × Female,-0.022,-0.045,-0.013
NH White × Female,-0.017,-0.028,-0.011
NH White × Male,-0.014,-0.017,-0.016
NH Black × Male,0.009,0.008,0.012
NH Other × Male,0.015,0.014,0.104
NH Asian × Female,0.026,0.008,0.005


### Finding: the TPR gap opens at both ends

- **Every female group's TPR falls in every model, Hispanic men's TPR falls in every model, and NH Asian men's TPR rises in every model.** The same pattern in three different models means it comes from the data, not from one model.
- **EOD here is exactly the TPR gap between two groups**, and both ends move apart:

| Model | 2014 | 2017 |
|---|---|---|
| LR | NH Asian × Male 0.847 − Hispanic × Female 0.323 = 0.524 | 0.879 − 0.301 = 0.578 |
| XGBoost | NH White × Male 0.837 − Hispanic × Female 0.368 = 0.469 | NH Asian × Male 0.863 − 0.355 = 0.508 |
| LR mitigated | NH Other × Female 0.721 − Hispanic × Male 0.570 = 0.151 | NH Asian × Male 0.730 − Hispanic × Male 0.519 = 0.211 |

- **Mitigated LR's EOD rise is real.** It is driven by Hispanic men, whose TPR falls from 0.570 to 0.519. This is a large group (about 900 people earning over $50K in the 2014 test set, SE ≈ 0.016), so −0.051 is about 3 × SE, and the same drop appears in all three models. This settles the question left open in Phase 2.
- NH Other × Female's −0.175 looks large, but the group is tiny; it only removes that group from the top of the range.
- **Small groups:** NH Other and NH Asian have only about 40–60 people earning over $50K in the 2014 test set, so their changes below about 0.05 should be ignored.

## Answer to RQ2: why does fairness change?

**Concept drift meets a fixed cutoff.**

1. **The inputs barely change** (Section 2: every PSI < 0.02).
2. **The models use them the same way every year** (Section 3: SHAP importance flat).
3. **But more people with the same features now earn over $50K,** mostly through real income growth, not inflation (Section 1: about 84% real).
4. **The newly positive people in low-scored groups** (women, Hispanic and NH Black workers) mostly score below the 2014 cutoff, so they are missed and those groups' TPR falls, while NH Asian men's TPR rises. The TPR gap opens at both ends, which raises EOD (Section 4).

**LR vs. XGBoost:** the same mechanism works in both, in the same direction for every group. LR's TPR falls slightly more in the low groups (by about 0.01–0.02), consistent with its heavy reliance on fixed occupation coefficients, but the difference is small and mostly within noise.

**Implication for Phase 4:** the problem sits at the **decision threshold**, not in the ranking (AUC is flat). That suggests testing threshold adjustment (one global cutoff, or one per group with Fairlearn's `ThresholdOptimizer`) against retraining.

**Limitations:**
- SHAP uses 1,000 rows per year, so changes below about 0.05 cannot be detected.
- Feature drift is measured one feature at a time; changes in how features combine (for example, education within occupation) would not show up in PSI.
- The inflation share uses national CPI-U, not Texas prices, and is likely a lower bound (round-number reporting).
- The 2014 baseline is the 25K-row test set, so small groups' 2014 TPRs are noisy.